# Xây Dựng & Chuẩn Hóa Bảng Lịch Sử Biến Động: `job_snapshots`

Notebook này tập trung xây dựng bảng lịch sử trạng thái theo thời gian **`job_snapshots`** trong kiến trúc Silver Layer:
- Áp dụng mô hình Daily Periodic Snapshot / SCD Type 2 để lưu giữ trạng thái của tin tuyển dụng tại từng mốc thời gian.
- Phục vụ các bài toán phân tích time-series chuyên sâu: biến động khung lương (salary trend), thời gian tồn tại trung bình của tin tuyển dụng (time-to-fill / job lifespan), và tỷ lệ tin hoạt động theo thời gian thực.
- Kiểm tra nghiêm ngặt ràng buộc DDL PostgreSQL: khóa chính `snapshot_id`, khóa ngoại tham chiếu `jobs(job_id)`, và ràng buộc duy nhất `UNIQUE(job_id, snapshot_date)`.

In [1]:
import os
import re
import json
from collections import Counter
from datetime import datetime, date
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Môi trường xử lý bảng job_snapshots đã sẵn sàng!')

Môi trường xử lý bảng job_snapshots đã sẵn sàng!


## 1. Nạp Dữ Liệu Bronze & Đồng Bộ Với Bảng `jobs`

Đồng bộ thông tin mức lương và thời gian thu thập từ bảng `jobs` (`02_jobs.ipynb`).

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
jobs_nb_path = os.path.abspath('02_jobs.ipynb')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)
df_merged['job_id'] = df_merged.index + 1

with open(jobs_nb_path, 'r', encoding='utf-8') as f:
    nb2 = json.load(f)
scope2 = {'pd': pd, 're': re}
exec(''.join(nb2['cells'][7]['source']), scope2)
parse_salary = scope2['parse_salary']

sal_records = [parse_salary(s) for s in df_merged['salary_raw_list']]
df_sal = pd.DataFrame(sal_records)
df_merged['salary_min_vnd'] = df_sal['salary_min_vnd']
df_merged['salary_max_vnd'] = df_sal['salary_max_vnd']

print(f'Tổng số tin tuyển dụng đồng bộ: {len(df_merged):,} tin')
print(f'Số tin có thông số lương cụ thể: {df_merged["salary_max_vnd"].notna().sum():,} tin')
print(f'Phân bố ngày crawl trên tập dữ liệu: {Counter(df_merged["raw_date_list"])}')

🧪 Kiểm tra thử nghiệm parser lương:
  • "25 - 40 triệu" -> Min: 25000000.0 | Max: 40000000.0 | Type: range | Curr: VND
  • "$ 1,000-2,500 /tháng" -> Min: 25400000.0 | Max: 63500000.0 | Type: range | Curr: USD
  • "Tới 30 triệu" -> Min: None | Max: 30000000.0 | Type: max_only | Curr: VND
  • "15000000" -> Min: 15000000.0 | Max: 15000000.0 | Type: fixed | Curr: VND
  • "14000000 - 18000000 MONTH" -> Min: 14000000.0 | Max: 18000000.0 | Type: range | Curr: VND
  • "$ 400-1,500 /tháng" -> Min: 10160000.0 | Max: 38100000.0 | Type: range | Curr: USD
  • "Thương lượng" -> Min: None | Max: None | Type: negotiable | Curr: VND
Tổng số tin tuyển dụng đồng bộ: 2,327 tin
Số tin có thông số lương cụ thể: 982 tin
Phân bố ngày crawl trên tập dữ liệu: Counter({'2026-10-11': 1510, '2026-10-12': 817})


## 2. Thiết Kế Pipeline Tạo Snapshot Theo Ngày (Periodic Snapshot Pipeline)

Tạo bản ghi snapshot theo chu kỳ hàng ngày:
- Ngày 2026-10-11: Chụp trạng thái đợt crawl 1 (1,510 tin).
- Ngày 2026-10-12: Chụp trạng thái toàn bộ tin tiếp tục tồn tại trên hệ thống và tin mới xuất hiện (2,327 tin).
- Xác định trạng thái `is_active` dựa trên hạn nộp hồ sơ (`deadline_raw`).

In [3]:
snapshots = []
snapshot_id_counter = 1
now_iso = datetime.now().isoformat()

# 1. Snapshot Dot 1: Ngay 2026-10-11
df_day1 = df_merged[df_merged['raw_date_list'] == '2026-10-11']
for idx, row in df_day1.iterrows():
    job_id = row['job_id']
    snap_date = '2026-10-11'
    dl_str = str(row.get('deadline_raw', '') or '')[:10]
    is_act = True if (not dl_str or dl_str >= snap_date) else False
    
    snapshots.append({
        'snapshot_id': snapshot_id_counter,
        'job_id': job_id,
        'snapshot_date': snap_date,
        'salary_min_vnd': row['salary_min_vnd'],
        'salary_max_vnd': row['salary_max_vnd'],
        'is_active': is_act,
        'created_at': now_iso
    })
    snapshot_id_counter += 1

# 2. Snapshot Dot 2: Ngay 2026-10-12
for idx, row in df_merged.iterrows():
    job_id = row['job_id']
    snap_date = '2026-10-12'
    dl_str = str(row.get('deadline_raw', '') or '')[:10]
    is_act = True if (not dl_str or dl_str >= snap_date) else False
    
    snapshots.append({
        'snapshot_id': snapshot_id_counter,
        'job_id': job_id,
        'snapshot_date': snap_date,
        'salary_min_vnd': row['salary_min_vnd'],
        'salary_max_vnd': row['salary_max_vnd'],
        'is_active': is_act,
        'created_at': now_iso
    })
    snapshot_id_counter += 1

df_job_snapshots = pd.DataFrame(snapshots)

print('Khởi tạo bảng job_snapshots thành công:')
print(f'  - Tổng số bản ghi snapshot: {len(df_job_snapshots):,}')
print(f'  - Tính duy nhất của Primary Key snapshot_id: {df_job_snapshots["snapshot_id"].is_unique}')
print(f'  - Tính duy nhất của Unique Constraint (job_id, snapshot_date): {len(df_job_snapshots) == len(df_job_snapshots.drop_duplicates(subset=["job_id", "snapshot_date"]))}')
print(f'  - Toàn vẹn khóa ngoại job_id: {df_job_snapshots["job_id"].between(1, len(df_merged)).all()} (100.0% hợp lệ)')
print(f'  - Tỷ lệ tin hoạt động (is_active = True): {(df_job_snapshots["is_active"] == True).sum():,} / {len(df_job_snapshots)} ({(df_job_snapshots["is_active"] == True).mean()*100:.1f}%)')

Khởi tạo bảng job_snapshots thành công:
  - Tổng số bản ghi snapshot: 3,837
  - Tính duy nhất của Primary Key snapshot_id: True
  - Tính duy nhất của Unique Constraint (job_id, snapshot_date): True
  - Toàn vẹn khóa ngoại job_id: True (100.0% hợp lệ)
  - Tỷ lệ tin hoạt động (is_active = True): 3,809 / 3837 (99.3%)


## 3. Đánh Giá Tính Toàn Vẹn Ràng Buộc & Chất Lượng Dữ Liệu Time-Series

In [4]:
valid_sal = df_job_snapshots[df_job_snapshots['salary_min_vnd'].notna() & df_job_snapshots['salary_max_vnd'].notna()]
is_sal_logical = (valid_sal['salary_min_vnd'] <= valid_sal['salary_max_vnd']).all()
is_date_valid = df_job_snapshots['snapshot_date'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()

print('--- KIỂM TRA LOGIC DỮ LIỆU BẢNG job_snapshots ---')
print(f'Logic lương tối thiểu <= lương tối đa: {is_sal_logical} (Kiểm tra trên {len(valid_sal):,} bản ghi có đủ 2 đầu lương)')
print(f'Format ngày snapshot chuẩn ISO YYYY-MM-DD: {is_date_valid}')

--- KIỂM TRA LOGIC DỮ LIỆU BẢNG job_snapshots ---
Logic lương tối thiểu <= lương tối đa: True (Kiểm tra trên 1,326 bản ghi có đủ 2 đầu lương)
Format ngày snapshot chuẩn ISO YYYY-MM-DD: True


## 4. Phân Tích Thống Kê Biến Động Theo Thời Gian & Xem Trước Bảng `job_snapshots`

In [5]:
print('--- 1. CẤU TRÚC BẢNG job_snapshots (INFO) ---')
print(df_job_snapshots.info())

print('\n--- 2. PHÂN BỐ BẢN GHI SNAPSHOT THEO NGÀY ---')
date_counts = df_job_snapshots['snapshot_date'].value_counts().sort_index()
df_date_summary = pd.DataFrame({'Số snapshot': date_counts, 'Tỷ lệ (%)': (date_counts / len(df_job_snapshots) * 100).round(2)})
print(df_date_summary.to_string())

print('\n--- 3. THỐNG KÊ MỨC LƯƠNG TRUNG BÌNH THEO TỪNG NGÀY SNAPSHOT ---')
sal_by_date = df_job_snapshots.groupby('snapshot_date')[['salary_min_vnd', 'salary_max_vnd']].mean().map(lambda x: f'{x:,.0f} đ' if pd.notna(x) else '')
print(sal_by_date.to_string())

print('\n--- 4. PREVIEW 10 DÒNG BẢNG job_snapshots (CHUẨN POSTGRESQL) ---')
preview_cols = ['snapshot_id', 'job_id', 'snapshot_date', 'salary_min_vnd', 'salary_max_vnd', 'is_active']
print(df_job_snapshots[preview_cols].head(10).to_string(index=False))

--- 1. CẤU TRÚC BẢNG job_snapshots (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 3837 entries, 0 to 3836
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   snapshot_id     3837 non-null   int64  
 1   job_id          3837 non-null   int64  
 2   snapshot_date   3837 non-null   str    
 3   salary_min_vnd  1326 non-null   float64
 4   salary_max_vnd  1593 non-null   float64
 5   is_active       3837 non-null   bool   
 6   created_at      3837 non-null   str    
dtypes: bool(1), float64(2), int64(2), str(2)
memory usage: 318.6 KB
None

--- 2. PHÂN BỐ BẢN GHI SNAPSHOT THEO NGÀY ---
               Số snapshot  Tỷ lệ (%)
snapshot_date                        
2026-10-11            1510      39.35
2026-10-12            2327      60.65

--- 3. THỐNG KÊ MỨC LƯƠNG TRUNG BÌNH THEO TỪNG NGÀY SNAPSHOT ---
              salary_min_vnd salary_max_vnd
snapshot_date                              
2026-10-11      22,771,808 